In [1]:
import os
import pandas as pd
import SimpleITK as sitk
from rt_utils import RTStructBuilder
from radiomics import featureextractor
import logging
import warnings
import pydicom

# Suppress warnings
warnings.filterwarnings("ignore")
logger = logging.getLogger("radiomics")
logger.setLevel(logging.ERROR)

# --- CONFIGURATION ---
source_name = 'Square'
source_path = '../Data/SQUARE_F/'
output_csv = '../Results/square_radiomics_clean.csv'

# Setup Extractor
params = {}
extractor = featureextractor.RadiomicsFeatureExtractor(**params)
extractor.settings['binWidth'] = 25
extractor.settings['resampledPixelSpacing'] = [1, 1, 1]
extractor.settings['interpolator'] = sitk.sitkBSpline
extractor.enableAllImageTypes()

print(f"--- STARTING SQUARE RADIOMICS (CLEAN MODE) ---")

# --- HELPER: Find Tumor ---
def find_target_roi(rtstruct):
    try:
        rois = rtstruct.get_roi_names()
    except: return None
    
    # Square Priorities
    target_priorities = ['ptv', 'ptv_total', 'ptv total', 'gtv', 'gtv_t']
    
    avail_clean = {name.lower().strip().replace(' ','').replace('_','').replace('-',''): name for name in rois}
    
    for priority in target_priorities:
        clean_p = priority.replace(' ','').replace('_','').replace('-','')
        for clean_a, original_name in avail_clean.items():
            if clean_p in clean_a:
                return original_name
    return None

# --- MAIN LOOP ---
results = []
patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])

for i, patient_id in enumerate(patient_folders):
    patient_dir = os.path.join(source_path, patient_id)
    ct_folder = os.path.join(patient_dir, 'CT')
    struct_folder = os.path.join(patient_dir, 'Struct')
    
    # Check if cleaning happened
    if not os.path.exists(ct_folder) or not os.path.exists(struct_folder):
        print(f"[{i+1}] {patient_id}: [SKIP] Clean folders not found. Run Cleaner script!")
        continue
        
    # Find RTStruct file inside the new /Struct folder
    rt_path = None
    if len(os.listdir(struct_folder)) > 0:
        rt_path = os.path.join(struct_folder, os.listdir(struct_folder)[0])
            
    if not rt_path:
        print(f"[{i+1}] {patient_id}: [SKIP] No Struct file found")
        continue

    try:
        # 1. Build RTStruct
        # Now we point specifically to the clean folders
        rtstruct = RTStructBuilder.create_from(
            dicom_series_path=ct_folder, 
            rt_struct_path=rt_path
        )
        
        # 2. Find ROI
        roi_name = find_target_roi(rtstruct)
        
        if roi_name:
            # 3. Mask & Extract
            mask_3d = rtstruct.get_roi_mask_by_name(roi_name)
            
            # Read image using SITK
            reader = sitk.ImageSeriesReader()
            dicom_names = reader.GetGDCMSeriesFileNames(ct_folder)
            reader.SetFileNames(dicom_names)
            image_sitk = reader.Execute()
            
            # Convert mask to SITK
            mask_sitk = sitk.GetImageFromArray(mask_3d.astype(int).transpose(2, 0, 1))
            mask_sitk.CopyInformation(image_sitk)
            
            features = extractor.execute(image_sitk, mask_sitk)
            
            row = {'PatientID': patient_id, 'Source': source_name, 'ROI_Name': roi_name}
            for k, v in features.items():
                if 'diagnostics' not in k: row[k] = v
            results.append(row)
            print(f"[{i+1}] {patient_id}: Success ({roi_name})")
        else:
            print(f"[{i+1}] {patient_id}: [SKIP] No Tumor Found")
            
    except Exception as e:
        print(f"[{i+1}] {patient_id}: [ERROR] {str(e)}")

# Save
if results:
    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)
    print(f"\nSUCCESS! Saved {len(df)} patients.")
else:
    print("\nNo data extracted.")

--- STARTING SQUARE RADIOMICS (CLEAN MODE) ---
[1] R130505087: Success (PTV)
[2] R1406007291: Success (PTV)
[3] R1508007367: Success (PTV)
[4] R1701004331: Success (PTV)
[5] R1702006414: Success (PTV)
[6] R1707004823: Success (PTV)
[7] R1710009613: Success (PTV)
[8] R1803003706: Success (PTV)
[9] R1807000334: Success (PTV)
[10] R1807003460: Success (PTV 60)
[11] R1807005275: Success (PTV)
[12] R1807007064: Success (PTV_Lung)
[13] R1808004062: Success (PTV60)
[14] R1810004752: Success (PTV)
[15] R1810006592: Success (PTV)
[16] R1810007735: Success (PTV)
[17] R1811001641: Success (PTV)
[18] R1903001183: Success (PTV)
[19] R1903004587: Success (PTV)
[20] R1905000571: Success (PTV)
[21] R1905002341: Success (PTV)
[22] R1905004532: Success (PTV lung)
[23] R1907005733: Success (PTV)
[24] R1907009307: Success (PTV)
[25] R1908007052: Success (PTV)
[26] R1908007690: Success (PTV)
[27] R1909004391: Success (PTV)
[28] R1909005566: Success (PTV)
[29] R1910000632: Success (PTV)
[30] R1910001778: Su